# Feature Engineering & Modeling Pipeline
## Limpieza, Feature Engineering, y Modelado con LightGBM

**Flujo general:**
1. Cargar datos y limpiar
2. Feature Engineering (features agregadas por cliente, establecimiento, giro, par)
3. Función `build_dataset()` para construir datasets de train/test
4. Entrenar modelo LightGBM con validación cruzada
5. Generar predicciones

---
## 1. CARGA DE DATOS

In [26]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# Cargar datos
train = pd.read_csv('01dataBaseTrainTrxRec.csv')
perfil = pd.read_csv('02dataBasePerfilRec.csv')
test_key = pd.read_csv('05dataBaseTestKeyRec.csv')

print(f"Train: {train.shape}")
print(f"Perfil: {perfil.shape}")
print(f"Test Key: {test_key.shape}")

Train: (1591617, 8)
Perfil: (30000, 14)
Test Key: (467203, 2)


In [28]:
train.head()

,fechaOper,codCliente,codGiro,codEstab,flagLimaProvEstab,ubigeoEstab,ctdTrx,ratingMonto
0,2017-01-29 00:00:00,7649,138.0,43629,1,176.0,1,0.014072
1,2016-12-01 00:00:00,24604,75.0,4326,0,81.0,1,0.001667
2,2017-06-01 00:00:00,15289,75.0,4326,0,81.0,1,0.000127
3,2017-09-22 00:00:00,5190,110.0,59776,1,156.0,1,0.001167
4,2017-05-05 00:00:00,16635,75.0,31043,0,81.0,1,0.047386


In [29]:
train['fechaOper'] = pd.to_datetime(train['fechaOper'])
train['fechaOper'] = pd.to_datetime(train['fechaOper'])
train['mes'] = train['fechaOper'].dt.month
train['dia_semana'] = train['fechaOper'].dt.dayofweek
train['es_finde'] = train['dia_semana'].isin([5, 6]).astype(int)
train['es_diciembre'] = (train['mes'] == 12).astype(int)
train['es_julio'] = (train['mes'] == 7).astype(int)

In [30]:
train.head()

,fechaOper,codCliente,codGiro,codEstab,flagLimaProvEstab,ubigeoEstab,ctdTrx,ratingMonto,mes,dia_semana,es_finde,es_diciembre,es_julio
0,2017-01-29,7649,138.0,43629,1,176.0,1,0.014072,1,6,1,0,0
1,2016-12-01,24604,75.0,4326,0,81.0,1,0.001667,12,3,0,1,0
2,2017-06-01,15289,75.0,4326,0,81.0,1,0.000127,6,3,0,0,0
3,2017-09-22,5190,110.0,59776,1,156.0,1,0.001167,9,4,0,0,0
4,2017-05-05,16635,75.0,31043,0,81.0,1,0.047386,5,4,0,0,0


---
## 2. LIMPIEZA — TRAIN

In [ ]:
# 2.1: Convertir fechas y extraer features temporales
train['fechaOper'] = pd.to_datetime(train['fechaOper'])
train['mes'] = train['fechaOper'].dt.month
train['dia_semana'] = train['fechaOper'].dt.dayofweek
train['es_finde'] = train['dia_semana'].isin([5, 6]).astype(int)
train['es_diciembre'] = (train['mes'] == 12).astype(int)
train['es_julio'] = (train['mes'] == 7).astype(int)

In [ ]:
# 2.2: Rellenar nulos en features categóricas
train['codGiro'] = train['codGiro'].fillna(0).astype(int)
train['ubigeoEstab'] = train['ubigeoEstab'].fillna(0).astype(int)

In [ ]:
# 2.3: Crear target (log transform de ratingMonto)
train['target'] = np.log1p(train['ratingMonto'])
print(f"Target (log): min={train['target'].min():.4f}, max={train['target'].max():.4f}")

---
## 3. LIMPIEZA — PERFIL

In [ ]:
# 3.1: saldoTcEntidad: nulo = sin tarjeta en esa entidad
for col in ['saldoTcEntidad1','saldoTcEntidad2','saldoTcEntidad3','saldoTcEntidad4']:
    perfil[col] = perfil[col].fillna('SinSaldo')

In [ ]:
# 3.2: Rellenar rangos ordinales con moda
for col in ['rangoIngreso', 'rangoEdad']:
    perfil[col] = perfil[col].fillna(perfil[col].mode()[0])

perfil['ubigeoCliente'] = perfil['ubigeoCliente'].fillna(perfil['ubigeoCliente'].mode()[0])
print("Nulos restantes en perfil:", perfil.isnull().sum().sum())

In [ ]:
# 3.3: Encoding ordinal (mapeo correcto, NO LabelEncoder)
rango_map = {'Rango1':1,'Rango2':2,'Rango3':3,'Rango4':4,'Rango5':5,'Rango6':6}
saldo_map = {'SinSaldo':0,'Rango1':1,'Rango2':2,'Rango3':3,'Rango4':4,'Rango5':5,'Rango6':6}

for col in ['rangoEdad','rangoIngreso','rangoCtdProdAct','rangoCtdProdPas','rangoCtdProdSeg']:
    perfil[col] = perfil[col].map(rango_map)

for col in ['saldoTcEntidad1','saldoTcEntidad2','saldoTcEntidad3','saldoTcEntidad4']:
    perfil[col] = perfil[col].map(saldo_map)

print("Encoding completado")

---
## 4. MERGE — TRAIN Y TEST_KEY

In [ ]:
train = train.merge(perfil, on='codCliente', how='left')
test_key = test_key.merge(perfil, on='codCliente', how='left')

print(f"Train limpio: {train.shape}")
print(f"Nulos en train: {train.isnull().sum().sum()}")
print(f"Nulos en test_key: {test_key.isnull().sum().sum()}")

---
## 5. FEATURE ENGINEERING — AGREGACIONES

In [ ]:
# 5.1: FEATURES POR CLIENTE
client_feats = train.groupby('codCliente').agg(
    # Actividad general
    total_trx_cliente        = ('ctdTrx', 'sum'),
    avg_trx_cliente          = ('ctdTrx', 'mean'),
    total_estab_visitados    = ('codEstab', 'nunique'),
    total_giros_visitados    = ('codGiro', 'nunique'),
    # Rating del cliente
    avg_rating_cliente       = ('ratingMonto', 'mean'),
    max_rating_cliente       = ('ratingMonto', 'max'),
    std_rating_cliente       = ('ratingMonto', 'std'),
    # Preferencia geográfica
    pct_lima_estab_cliente   = ('flagLimaProvEstab', 'mean'),
    # Temporalidad
    meses_activo             = ('mes', 'nunique'),
    pct_finde_cliente        = ('es_finde', 'mean'),
).reset_index()

print(f"Client features: {client_feats.shape}")

In [ ]:
# 5.2: FEATURES POR ESTABLECIMIENTO
estab_feats = train.groupby('codEstab').agg(
    # Popularidad
    total_clientes_estab     = ('codCliente', 'nunique'),
    total_trx_estab          = ('ctdTrx', 'sum'),
    avg_trx_estab            = ('ctdTrx', 'mean'),
    # Rating del establecimiento
    avg_rating_estab         = ('ratingMonto', 'mean'),
    max_rating_estab         = ('ratingMonto', 'max'),
    std_rating_estab         = ('ratingMonto', 'std'),
    # Temporalidad
    pct_finde_estab          = ('es_finde', 'mean'),
    meses_activo_estab       = ('mes', 'nunique'),
).reset_index()

print(f"Estab features: {estab_feats.shape}")

In [ ]:
# 5.3: FEATURES POR GIRO (rubro)
giro_feats = train.groupby('codGiro').agg(
    avg_rating_giro          = ('ratingMonto', 'mean'),
    total_clientes_giro      = ('codCliente', 'nunique'),
    total_trx_giro           = ('ctdTrx', 'sum'),
    popularidad_giro         = ('codEstab', 'nunique'),
).reset_index()

print(f"Giro features: {giro_feats.shape}")

In [ ]:
# 5.4: TARGET ENCODING por codGiro y ubigeoEstab
giro_target = train.groupby('codGiro')['ratingMonto'].mean().reset_index()
giro_target.columns = ['codGiro', 'target_enc_giro']

ubigeo_target = train.groupby('ubigeoEstab')['ratingMonto'].mean().reset_index()
ubigeo_target.columns = ['ubigeoEstab', 'target_enc_ubigeo']

print(f"Giro target encoding: {giro_target.shape}")
print(f"Ubigeo target encoding: {ubigeo_target.shape}")

In [ ]:
# 5.5: FEATURES POR PAR CLIENTE-ESTABLECIMIENTO (interacción directa)
pair_feats = train.groupby(['codCliente', 'codEstab']).agg(
    trx_par                  = ('ctdTrx', 'sum'),
    avg_rating_par           = ('ratingMonto', 'mean'),
    max_rating_par           = ('ratingMonto', 'max'),
    visitas_par              = ('ratingMonto', 'count'),
    meses_visitado_par       = ('mes', 'nunique'),
    pct_finde_par            = ('es_finde', 'mean'),
).reset_index()

print(f"Pair features: {pair_feats.shape}")

In [ ]:
# 5.6: COLD START — establecimientos sin historial
# Asociar giro a cada establecimiento y usar promedio del giro
estab_giro = train[['codEstab','codGiro']].dropna().drop_duplicates('codEstab')
estab_feats = estab_feats.merge(estab_giro, on='codEstab', how='left')
estab_feats = estab_feats.merge(
    giro_feats[['codGiro','avg_rating_giro','total_clientes_giro']],
    on='codGiro', how='left'
)

print("Cold start setup completado")

In [ ]:
# 5.7: FEATURES CLIENTE-GIRO (qué rubros prefiere cada cliente)
client_giro_feats = train.groupby(['codCliente', 'codGiro']).agg(
    trx_cliente_giro         = ('ctdTrx', 'sum'),
    avg_rating_cliente_giro  = ('ratingMonto', 'mean'),
).reset_index()

# Rubro favorito del cliente (el de mayor rating promedio)
giro_favorito = client_giro_feats.loc[
    client_giro_feats.groupby('codCliente')['avg_rating_cliente_giro'].idxmax()
][['codCliente', 'codGiro']].rename(columns={'codGiro': 'giro_favorito'})

client_feats = client_feats.merge(giro_favorito, on='codCliente', how='left')

print("Giro favorito agregado a client_feats")

In [ ]:
# 5.8: ACTIVIDAD RECIENTE (últimos 3 meses)
fecha_max = train['fechaOper'].max()
train['es_reciente'] = (train['fechaOper'] >= fecha_max - pd.DateOffset(months=3)).astype(int)

pair_reciente = train[train['es_reciente']==1].groupby(['codCliente','codEstab']).agg(
    avg_rating_par_reciente = ('ratingMonto', 'mean'),
    trx_par_reciente        = ('ctdTrx', 'sum'),
).reset_index()

print(f"Pair reciente: {pair_reciente.shape}")

In [ ]:
# 5.9: TENDENCIA (primera vs segunda mitad del historial)
train_sorted = train.sort_values(['codCliente','codEstab','fechaOper'])
train_sorted['cumidx'] = train_sorted.groupby(['codCliente','codEstab']).cumcount()
train_sorted['total_v'] = train_sorted.groupby(['codCliente','codEstab'])['codEstab'].transform('count')
train_sorted['es_segunda_mitad'] = (train_sorted['cumidx'] >= train_sorted['total_v']/2).astype(int)

tendencia = train_sorted.groupby(['codCliente','codEstab','es_segunda_mitad'])['ratingMonto'].mean().unstack()
tendencia.columns = ['rating_primera_mitad','rating_segunda_mitad']
tendencia['tendencia_rating'] = tendencia['rating_segunda_mitad'] - tendencia['rating_primera_mitad']
tendencia = tendencia.reset_index()[['codCliente','codEstab','tendencia_rating']]

print(f"Tendencia: {tendencia.shape}")

---
## 6. FUNCIÓN `build_dataset()` — SECUENCIA CRÍTICA

In [ ]:
def build_dataset(df, client_feats, estab_feats, pair_feats):
    """
    Construye dataset final mergeando features agregadas.
    
    SECUENCIA DE MERGES:
    1. giro_target (1-a-1)
    2. client_feats (1-a-1) — clientPerfil ya está en df
    3. estab_feats (1-a-1)
    4. pair_feats (1-a-1) — features de interacción cliente-estab
    5. pair_reciente (1-a-1) — últimos 3 meses
    6. tendencia (1-a-1) — cambio rating en el tiempo
    7. ubigeo_target (1-a-1)
    
    RELLENOS (fillna):
    - pair_reciente: 0 (sin transacciones recientes)
    - tendencia: 0 (sin tendencia observable)
    - target_enc_giro, target_enc_ubigeo: 0.0132 (media global)
    
    RATIOS Y DIFERENCIAS:
    - ratio_estab_sobre_cliente_log (log para manejar extremos)
    - diff_rating_par_vs_cliente
    - diff_rating_par_vs_estab
    """
    # ─────────────────────────────────────────────────────────
    # PASO 1: Merges base (features agregadas)
    # ─────────────────────────────────────────────────────────
    df = df.merge(giro_target,   on='codGiro',   how='left')
    df = df.merge(client_feats,  on='codCliente', how='left')
    df = df.merge(estab_feats,   on='codEstab',   how='left')
    df = df.merge(pair_feats,    on=['codCliente','codEstab'], how='left')
    
    # ─────────────────────────────────────────────────────────
    # PASO 2: Merges de features nuevas (reciente, tendencia, ubigeo)
    # ─────────────────────────────────────────────────────────
    df = df.merge(pair_reciente, on=['codCliente','codEstab'], how='left')
    df = df.merge(tendencia,     on=['codCliente','codEstab'], how='left')
    df = df.merge(ubigeo_target, on='ubigeoEstab', how='left')
    
    # ─────────────────────────────────────────────────────────
    # PASO 3: Rellenar NaNs (fillna)
    # ─────────────────────────────────────────────────────────
    df['avg_rating_par_reciente'] = df['avg_rating_par_reciente'].fillna(0)
    df['trx_par_reciente']        = df['trx_par_reciente'].fillna(0)
    df['tendencia_rating']        = df['tendencia_rating'].fillna(0)
    df['target_enc_giro']         = df['target_enc_giro'].fillna(0.0132)
    df['target_enc_ubigeo']       = df['target_enc_ubigeo'].fillna(0.0132)
    
    # ─────────────────────────────────────────────────────────
    # PASO 4: Transformaciones adicionales
    # ─────────────────────────────────────────────────────────
    # (Nota: ratio_estab_sobre_cliente debe existir previamente)
    if 'ratio_estab_sobre_cliente' in df.columns:
        df['ratio_estab_sobre_cliente_log'] = np.log1p(df['ratio_estab_sobre_cliente'])
    
    # ─────────────────────────────────────────────────────────
    # PASO 5: Calcular diferencias (comparaciones relativas)
    # ─────────────────────────────────────────────────────────
    df['diff_rating_par_vs_cliente'] = df['avg_rating_par'] - df['avg_rating_cliente']
    df['diff_rating_par_vs_estab']   = df['avg_rating_par'] - df['avg_rating_estab']
    
    return df

print("Función build_dataset() definida.")

---
## 7. CONSTRUCCIÓN DE DATASETS TRAIN Y TEST

In [ ]:
# Aplicar build_dataset() a train y test_key
X_train = build_dataset(train, client_feats, estab_feats, pair_feats)
#X_test  = build_dataset(test_key, client_feats, estab_feats, pair_feats)

print(f"X_train: {X_train.shape}")
#print(f"X_test: {X_test.shape}")
print(f"\nNulos en X_train: {X_train.isnull().sum().sum()}")
#print(f"Nulos en X_test: {X_test.isnull().sum().sum()}")

In [ ]:
# Seleccionar features para el modelo
drop_cols = [
    'codCliente', 'codEstab',
    'ratingMonto', 'target',
    'fechaOper', 'mes', 'dia_semana', 'es_finde', 'es_diciembre', 'es_julio',
    'codGiro_x', 'codGiro_y',
    'es_reciente',        # auxiliar
    'cumidx', 'total_v', 'es_segunda_mitad',  # auxiliares
]

# Verificar que todas las features existen en X_train
features = [c for c in X_train.columns if c in X_train.columns]

y_train = X_train['target']  # log1p(ratingMonto)

print(f"Features totales: {len(features)}")
print(f"Shape X_train[features]: {X_train[features].shape}")
print(f"\nLista de features ({len(features)} total):")
for f in sorted(features):
    print(f"  - {f}")

---
## 8. MODELO — LightGBM con Validación Cruzada

In [ ]:
# Configuración de validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X_train))
feat_importance = np.zeros(len(features))

# Parámetros de LightGBM
params = {
    'n_estimators'     : 500,
    'learning_rate'    : 0.03,
    'num_leaves'       : 63,
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'min_gain_to_split': 0.01,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

print("Iniciando validación cruzada...\n")

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f'\n===== Fold {fold+1}/5 =====')

    X_tr  = X_train[features].iloc[tr_idx]
    X_val = X_train[features].iloc[val_idx]
    y_tr  = y_train.iloc[tr_idx]
    y_val = y_train.iloc[val_idx]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(100)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    feat_importance += model.feature_importances_ / 5

print("\n" + "="*50)
print("Validación cruzada completada.")

In [ ]:
# Evaluar rendimiento
oof_rmse = np.sqrt(mean_squared_error(y_train, oof_preds))
print(f'\n✅ OOF RMSE (escala log): {oof_rmse:.6f}')

# Convertir a escala original
oof_real = np.expm1(oof_preds)
y_real = np.expm1(y_train)
oof_rmse_real = np.sqrt(mean_squared_error(y_real, oof_real))
print(f'✅ OOF RMSE (escala original): {oof_rmse_real:.6f}')

In [ ]:
# Feature importance
fi_df = pd.DataFrame({
    'feature'   : features,
    'importance': feat_importance
}).sort_values('importance', ascending=False)

print('\nTop 20 features más importantes:')
print(fi_df.head(20).to_string(index=False))

# Visualizar
plt.figure(figsize=(10, 8))
fi_df.head(20).plot(kind='barh', x='feature', y='importance', figsize=(10, 8), color='steelblue', legend=False)
plt.title('Feature Importance — LightGBM')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 9. PREDICCIONES EN TEST Y SUBMISSION

In [ ]:
# Reentrenar modelo final con TODOS los datos de train
print("Entrenando modelo final con todo el train...\n")

model_final = lgb.LGBMRegressor(**params)
model_final.fit(
    X_train[features], 
    y_train,
    callbacks=[lgb.log_evaluation(100)]
)

print("\n✅ Modelo final entrenado.")

In [ ]:
# Realizar predicciones
test_preds = model_final.predict(X_test[features])

# Invertir log1p y clipear al rango [0, 1]
test_preds = np.expm1(test_preds)
test_preds = np.clip(test_preds, 0, 1)

print(f"Predicciones realizadas: {len(test_preds)}")
print(f"Rango: [{test_preds.min():.6f}, {test_preds.max():.6f}]")
print(f"Media: {test_preds.mean():.6f}")

In [ ]:
# Generar submission
test = pd.read_csv('03dataBaseTestRec.csv')
test['ratingMonto'] = test_preds
test.to_csv('submission.csv', index=False)

print(f"\n✅ Submission generado: {len(test)} filas")
print(f"Guardado en: submission.csv")
print(f"\nPrimeras 10 filas:")
print(test.head(10))